# Delphi ONNX Inference

This notebook demonstrates how to:
1. Load the Delphi model in ONNX format
2. Create a patient health timeline
3. Convert data to the expected input format
4. Run inference and interpret results

## Prerequisites

```bash
pip install onnxruntime numpy pandas
```

In [69]:
import numpy as np
import pandas as pd
import onnxruntime as ort
from pathlib import Path

## 1. Load Labels and Create Token Mapping

Delphi uses 1,270 tokens representing:
- Special tokens: Padding (0), No event (1), Female (2), Male (3)
- Lifestyle: BMI (4-6), Smoking (7-9), Alcohol (10-12)
- Medical codes: ICD-10 codes (13-1268)
- Death (1269)

In [70]:
# Load labels - read as plain text since disease names contain commas
labels_path = Path("../data/ukb_simulated_data/labels.csv")

with open(labels_path, "r") as f:
    label_names = [line.strip() for line in f.readlines()]

labels = pd.DataFrame({"name": label_names})
labels["index"] = labels.index

# Create name -> token_id mapping
# Note: In the data, tokens are stored as 0-indexed, but Delphi adds +1 during batch creation
# So we need to add 1 to match the model's expected input
name_to_token = {row["name"]: row["index"] + 1 for _, row in labels.iterrows()}
token_to_name = {v: k for k, v in name_to_token.items()}

print(f"Loaded {len(labels)} labels")
print("\nSpecial tokens:")
for i in range(13):
    print(f"  {i+1}: {labels.iloc[i]['name']}")

Loaded 1270 labels

Special tokens:
  1: Padding
  2: No event
  3: Female
  4: Male
  5: BMI_low
  6: BMI_mid
  7: BMI_high
  8: Smoking_low
  9: Smoking_mid
  10: Smoking_high
  11: Alcohol_low
  12: Alcohol_mid
  13: Alcohol_high


## 2. Load the ONNX Model

In [71]:
# Load ONNX model
onnx_path = "delphi.onnx"
session = ort.InferenceSession(onnx_path)

# Check model inputs/outputs
print("Model inputs:")
for inp in session.get_inputs():
    print(f"  {inp.name}: {inp.type} {inp.shape}")

print("\nModel outputs:")
for out in session.get_outputs():
    print(f"  {out.name}: {out.type} {out.shape}")

Model inputs:
  idx: tensor(int64) ['batch_size', 'seq_len']
  age: tensor(float) ['batch_size', 'seq_len']

Model outputs:
  logits: tensor(float) ['batch_size', 'seq_len', 1270]


## 3. Create a Patient Health Timeline

A health timeline is a sequence of (event_name, age_in_years) tuples.

### Example Patient
- Male, born at age 0
- Had chickenpox at age 2
- Developed atopic dermatitis at age 3
- Migraine started at age 20
- Lactose intolerance at age 21
- Infectious mononucleosis at age 22
- Influenza at age 28
- At age 41: low smoking, mid BMI, low alcohol

In [72]:
def create_patient_timeline(events_with_ages):
    """
    Create a patient timeline from a list of (event_name, age_in_years) tuples.
    """
    tokens = []
    ages = []
    
    for event_name, age_years in events_with_ages:
        # Convert age from years to days
        age_days = age_years * 365.25
        
        # Look up token ID
        if event_name not in name_to_token:
            print(f"Warning: Unknown event '{event_name}', skipping")
            continue
            
        token_id = name_to_token[event_name]
        tokens.append(token_id)
        ages.append(age_days)
    
    return [np.array(tokens, dtype=np.int64), np.array(ages, dtype=np.float32)]

In [73]:
# Example patient timeline
example_timeline = [
    ("Male", 0),
    ("B01 (varicella [chickenpox])", 2),
    ("L20 (atopic dermatitis)", 3),
    ("No event", 5),
    ("No event", 10),
    ("No event", 15),
    ("No event", 20),
    ("G43 (migraine)", 20),
    ("E73 (lactose intolerance)", 21),
    ("B27 (infectious mononucleosis)", 22),
    ("No event", 25),
    ("J11 (influenza, virus not identified)", 28),
    ("No event", 30),
    ("No event", 35),
    ("No event", 40),
    ("Smoking_low", 41),
    ("BMI_mid", 41),
    ("Alcohol_low", 41),
    ("No event", 42),
]

tokens, ages = create_patient_timeline(example_timeline)

print("Patient timeline:")
print(f"  Sequence length: {len(tokens)}")
print(f"  Age range: {ages.min()/365.25:.1f} - {ages.max()/365.25:.1f} years")
print("\nEvents:")
for i, (tok, age) in enumerate(zip(tokens, ages)):
    print(f"  {i}: {token_to_name[tok]:50s} at age {age/365.25:.1f}")

Patient timeline:
  Sequence length: 19
  Age range: 0.0 - 42.0 years

Events:
  0: Male                                               at age 0.0
  1: B01 (varicella [chickenpox])                       at age 2.0
  2: L20 (atopic dermatitis)                            at age 3.0
  3: No event                                           at age 5.0
  4: No event                                           at age 10.0
  5: No event                                           at age 15.0
  6: No event                                           at age 20.0
  7: G43 (migraine)                                     at age 20.0
  8: E73 (lactose intolerance)                          at age 21.0
  9: B27 (infectious mononucleosis)                     at age 22.0
  10: No event                                           at age 25.0
  11: J11 (influenza, virus not identified)              at age 28.0
  12: No event                                           at age 30.0
  13: No event                        

## 4. Run ONNX Inference

In [74]:
from typing import List, Tuple

def run_inference(session, timeline: List[Tuple[str, int]]):
    """
    Run ONNX inference on a patient timeline.
    
    Returns:
        logits: numpy array of shape (seq_len, vocab_size)
    """
    # Add batch dimension
    tokens, ages = create_patient_timeline(timeline)
    idx = tokens.reshape(1, -1)
    age = ages.reshape(1, -1)
    
    # Run inference
    outputs = session.run(["logits"], {"idx": idx, "age": age})
    logits = outputs[0]
    
    # Remove batch dimension
    return logits[0]

# Run inference
logits = run_inference(session, timeline=example_timeline)

print(f"Output shape: {logits.shape}")
print(f"Logits range: [{logits.min():.2f}, {logits.max():.2f}]")

Output shape: (19, 1270)
Logits range: [-19.74, -6.71]


## 5. Interpret Predictions

The logits represent the log-rates for each possible next event.
We can:
1. Get the most likely next events
2. Calculate disease risk probabilities
3. Compute expected time to events

In [75]:
def get_top_predictions(logits, token_to_name, k=10, ignore_tokens=None):
    """
    Get the top-k most likely next events.
    
    Args:
        logits: logits for the last position, shape (vocab_size,)
        token_to_name: mapping from token ID to name
        k: number of top predictions to return
        ignore_tokens: list of token IDs to ignore (e.g., padding, lifestyle)
        
    Returns:
        List of (token_name, logit, rate) tuples
    """
    if ignore_tokens is None:
        # Ignore padding (1), gender (3-4), and lifestyle tokens (5-13)
        # Note: "No event" (token 2) is NOT ignored
        ignore_tokens = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
    
    # Create mask for valid tokens
    valid_mask = np.ones(len(logits), dtype=bool)
    for tok in ignore_tokens:
        if tok < len(logits):
            valid_mask[tok] = False
    
    # Get indices sorted by logit value (descending)
    sorted_indices = np.argsort(logits)[::-1]
    
    results = []
    for idx in sorted_indices:
        if valid_mask[idx] and idx in token_to_name:
            name = token_to_name[idx]
            logit = logits[idx]
            results.append((name, logit))
            if len(results) >= k:
                break
    
    return results

# Get predictions for the last position (predicting what happens after age 42)
last_logits = logits[-1]

print("Top 15 most likely next events after age 42:")
print("=" * 70)
top_preds = get_top_predictions(last_logits, token_to_name, k=15)

for i, (name, logit) in enumerate(top_preds, 1):
    print(f"{i:2d}. {name:50s} logit={logit:.2f}")

Top 15 most likely next events after age 42:
 1. E77 (disorders of glycoprotein metabolism)         logit=-11.79
 2. M24 (other specific joint derangements)            logit=-11.84
 3. M77 (other enthesopathies)                         logit=-11.99
 4. M53 (other dorsopathies, not elsewhere classified) logit=-11.99
 5. I09 (other rheumatic heart diseases)               logit=-12.05
 6. M18 (arthrosis of first carpometacarpal joint)     logit=-12.06
 7. L29 (pruritus)                                     logit=-12.32
 8. J44 (other chronic obstructive pulmonary disease)  logit=-12.35
 9. J05 (acute obstructive laryngitis [croup] and epiglottitis) logit=-12.38
10. J21 (acute bronchiolitis)                          logit=-12.38
11. J22 (unspecified acute lower respiratory infection) logit=-12.39
12. F16 (mental and behavioural disorders due to use of hallucinogens) logit=-12.51
13. B34 (viral infection of unspecified site)          logit=-12.58
14. K63 (other diseases of intestine)        

## What Does a Negative Logit Mean?

In the Delphi model, **logits represent log-rates** for an exponential time-to-event model:

```
logit = log(rate)
rate = exp(logit)
```

### Interpretation in Delphi Context

| Logit | Rate (events/day) | Expected Time | Meaning |
|-------|------------------|---------------|---------|
| -10 | 0.000045 | 22,026 days (60 years) | Very unlikely soon |
| -8 | 0.000335 | 2,985 days (8.2 years) | Unlikely soon |
| -5 | 0.0067 | 149 days (5 months) | Somewhat likely |
| 0 | 1.0 | 1 day | Baseline |
| 2 | 7.39 | 0.135 days (3.2 hours) | Very likely soon |
| 5 | 148 | 0.0068 days (9.8 minutes) | Extremely likely soon |


## 6. Calculate Disease Risk Over Time

We can compute the probability of developing a specific disease within a time window using the exponential distribution:

**P(event within t) = 1 - exp(-rate × t)**

In [76]:
def calculate_disease_risk(logits, disease_tokens, time_horizon_years=10):
    """
    Calculate the risk of developing specific diseases within a time horizon.
    
    Uses exponential distribution: P(event within t) = 1 - exp(-rate * t)
    
    Args:
        logits: logits for the last position
        disease_tokens: list of token IDs for diseases of interest
        time_horizon_years: time window in years
        
    Returns:
        dict mapping token_id to risk probability
    """
    time_horizon_days = time_horizon_years * 365.25
    risks = {}
    
    for tok in disease_tokens:
        if tok < len(logits):
            rate = np.exp(logits[tok])
            # Probability of event within time horizon
            risk = 1 - np.exp(-rate * time_horizon_days)
            risks[tok] = risk
    
    return risks

# Diseases of interest (common serious conditions)
diseases_of_interest = {
    "I10 (essential (primary) hypertension)": name_to_token.get("I10 (essential (primary) hypertension)"),
    "E11 (non-insulin-dependent diabetes mellitus)": name_to_token.get("E11 (non-insulin-dependent diabetes mellitus)"),
    "I25 (chronic ischaemic heart disease)": name_to_token.get("I25 (chronic ischaemic heart disease)"),
    "J44 (other chronic obstructive pulmonary disease)": name_to_token.get("J44 (other chronic obstructive pulmonary disease)"),
    "I50 (heart failure)": name_to_token.get("I50 (heart failure)"),
    "C34 Malignant neoplasm of bronchus and lung": name_to_token.get("C34 Malignant neoplasm of bronchus and lung"),
    "F32 (depressive episode)": name_to_token.get("F32 (depressive episode)"),
    "Death": name_to_token.get("Death"),
}

# Filter out None values
diseases_of_interest = {k: v for k, v in diseases_of_interest.items() if v is not None}

print("10-year disease risk predictions for this patient:")
print("=" * 60)

disease_tokens = list(diseases_of_interest.values())
risks = calculate_disease_risk(last_logits, disease_tokens, time_horizon_years=10)

for name, tok in diseases_of_interest.items():
    if tok in risks:
        risk_pct = risks[tok] * 100
        print(f"{name:50s}: {risk_pct:6.2f}%")

10-year disease risk predictions for this patient:
I10 (essential (primary) hypertension)            :   0.02%
E11 (non-insulin-dependent diabetes mellitus)     :   0.01%
I25 (chronic ischaemic heart disease)             :   0.18%
J44 (other chronic obstructive pulmonary disease) :   1.58%
I50 (heart failure)                               :   0.12%
C34 Malignant neoplasm of bronchus and lung       :   0.01%
F32 (depressive episode)                          :   0.10%


## 7. Generate Trajectory Until Death

Now we'll use the model to generate a complete life trajectory by iteratively sampling events until death or max age.

**Algorithm (Competing Exponentials):**
1. Get logits for all possible next events
2. Sample time-to-event for each: t_i = -log(u) / exp(logit_i)  
3. The event with minimum sampled time wins
4. Append event and advance age
5. Repeat until Death or max_age

In [77]:
def generate_trajectory_until_death(
    session,
    seed_tokens,
    seed_ages,
    token_to_name,
    *,
    max_new_tokens=50,
    max_age_years=85,
    ignore_tokens=None,
    death_token=None,
):
    """
    Generate a patient trajectory until death or max age using ONNX inference.
    
    Uses the competing exponentials sampling method:
    - For each possible event, sample time t_i = -exp(-logit_i) * log(u_i)
    - The event with minimum sampled time happens next
    - Continue until death token or max_age is reached
    """
    if ignore_tokens is None:
        # Default: ignore padding (1), gender (3-4), and lifestyle tokens (5-13)
        # Note: "No event" (token 2) is NOT ignored - it should be allowed!
        ignore_tokens = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
    
    max_age_days = max_age_years * 365.25
    
    # Start with seed - ensure correct dtypes
    tokens = seed_tokens.copy().astype(np.int64)
    ages = seed_ages.copy().astype(np.float32)
    
    for step in range(max_new_tokens):
        # Run inference - ensure correct dtypes for ONNX
        idx = tokens.reshape(1, -1).astype(np.int64)
        age = ages.reshape(1, -1).astype(np.float32)
        outputs = session.run(["logits"], {"idx": idx, "age": age})
        logits = outputs[0][0, -1, :]  # Last position logits
        
        # Mask ignored tokens
        for tok in ignore_tokens:
            if tok < len(logits):
                logits[tok] = -np.inf
        
        # Mask already-seen tokens (no repeat)
        for tok in tokens:
            if tok != 1 and tok < len(logits):  # Don't mask "No event"
                logits[tok] = -np.inf
        
        # Sample from competing exponentials
        # t_i = -log(u_i) / rate_i where rate_i = exp(logit_i)
        u = np.random.rand(len(logits))
        u = np.clip(u, 1e-10, 1.0)  # Avoid log(0)
        
        rates = np.exp(logits)
        t_next = -np.log(u) / np.clip(rates, 1e-10, None)
        t_next = np.clip(t_next, 0, 365 * 80)
        
        # Event with minimum time wins
        idx_next = int(np.argmin(t_next))
        dt = float(t_next[idx_next])
        age_next = float(ages[-1]) + dt
        
        # Append to sequence - maintain dtypes
        tokens = np.append(tokens, np.int64(idx_next))
        ages = np.append(ages, np.float32(age_next))
        
        # Check termination
        if death_token is not None and idx_next == death_token:
            break
        if age_next > max_age_days:
            break
    
    # Convert to event list
    events = []
    for tok, age_days in zip(tokens, ages):
        name = token_to_name.get(int(tok), f"Unknown({tok})")
        events.append((name, float(age_days) / 365.25))
    
    return tokens, ages, events

In [78]:
# Get death token
death_token = name_to_token.get("Death")
print(f"Death token ID: {death_token}")

# Generate trajectory from our example patientprint
# Diagnostic: Check death token logit
print("=" * 70)
print("DIAGNOSTICS:")
print("=" * 70)

# Get logits at the last position before death (or end)
last_idx = tokens.reshape(1, -1).astype(np.int64)
last_age = ages.reshape(1, -1).astype(np.float32)
last_outputs = session.run(["logits"], {"idx": last_idx, "age": last_age})
last_logits = last_outputs[0][0, -1, :]

vocab_size = len(last_logits)
if death_token > 0 and death_token <= vocab_size:
    death_logit_idx = death_token - 1
    death_logit = last_logits[death_logit_idx]
death_rate = np.exp(death_logit)
print(f"Death token ({death_token}) logit: {death_logit:.4f}")
print(f"Death rate (events/day): {death_rate:.6e}")
print(f"Expected time to death: {1/death_rate if death_rate > 0 else 'inf':.1f} days ({1/death_rate/365.25 if death_rate > 0 else 'inf':.1f} years)")

# Show top 10 logits for comparison
top_indices = np.argsort(last_logits)[-10:][::-1]
print("Top 10 logits (for comparison):")
for idx in top_indices:
    name = token_to_name.get(int(idx), f"Token {idx}")
    print(f"  {name:50s}: {last_logits[idx]:.4f}")

Death token ID: 1270
DIAGNOSTICS:
Death token (1270) logit: -15.6994
Death rate (events/day): 1.519933e-07
Expected time to death: 6579238.4 days (18013.0 years)
Top 10 logits (for comparison):
  Padding                                           : -7.6351
  E77 (disorders of glycoprotein metabolism)        : -11.7851
  M24 (other specific joint derangements)           : -11.8385
  M77 (other enthesopathies)                        : -11.9919
  M53 (other dorsopathies, not elsewhere classified): -11.9935
  I09 (other rheumatic heart diseases)              : -12.0475
  M18 (arthrosis of first carpometacarpal joint)    : -12.0626
  L29 (pruritus)                                    : -12.3202
  J44 (other chronic obstructive pulmonary disease) : -12.3452
  J05 (acute obstructive laryngitis [croup] and epiglottitis): -12.3779


## 8. Generate Multiple Trajectories

Since the model is stochastic (due to exponential sampling), we can generate multiple possible futures for the same patient to see the range of outcomes.

In [79]:
# Generate 5 different trajectories for the same patient
np.random.seed(None)  # Reset seed for randomness

n_trajectories = 5
trajectories = []

print("Generating multiple trajectories...")
print("=" * 70)

[tokens, ages] = create_patient_timeline(example_timeline)
for i in range(n_trajectories):
    tokens, ages, events = generate_trajectory_until_death(
        session,
        tokens,
        ages,
        token_to_name,
        max_new_tokens=60,
        max_age_years=85,
        death_token=death_token,
    )
    trajectories.append((tokens, ages, events))
    
    # Get final age and cause
    final_age = ages[-1] / 365.25
    final_event = events[-1][0]
    n_new = len(events) - len(tokens)
    
    print(f"Trajectory {i+1}:")
    print(f"  Final age: {final_age:.1f} years")
    print(f"  New events: {n_new}")
    print(f"  Ended with: {final_event}")
    
    # Show last 5 events
    print("  Last 10 events:")
    for name, age in events[-10:]:
        print(f"    - {name[:45]:45s} @ {age:.1f}")
    print(f"    - Death @ {final_age:.1f}")

Generating multiple trajectories...
Trajectory 1:
  Final age: 86.1 years
  New events: 0
  Ended with: H90 (conductive and sensorineural hearing loss)
  Last 10 events:
    - O01 Hydatidiform mole38                       @ 81.3
    - G55 (nerve root and plexus compressions in di @ 81.4
    - I82 (other venous embolism and thrombosis)    @ 81.4
    - O70 (perineal laceration during delivery)     @ 81.8
    - M77 (other enthesopathies)                    @ 82.5
    - E54 (ascorbic acid deficiency)                @ 82.9
    - M16 (coxarthrosis [arthrosis of hip])         @ 83.3
    - K75 (other inflammatory liver diseases)       @ 83.7
    - H22 (disorders of iris and ciliary body in di @ 84.8
    - H90 (conductive and sensorineural hearing los @ 86.1
    - Death @ 86.1
Trajectory 2:
  Final age: 86.1 years
  New events: 0
  Ended with: I50 (heart failure)
  Last 10 events:
    - G55 (nerve root and plexus compressions in di @ 81.4
    - I82 (other venous embolism and thrombosis)    @ 81